# Experimentos 🧪

Una parte clave de cualquier estudio de simulación por computadora es la **experimentación**.  Aquí se llevarán a cabo una serie de experimentos en un intento de comprender y encontrar mejoras al sistema en estudio.  Los experimentos esencialmente varían las entradas y la lógica del proceso.

Podemos hacer esto manualmente, pero a medida que desarrollemos un modelo, la cantidad de parámetros de entrada aumentará. 

💡 Existen varias estructuras de datos que puede emplear para organizar los parámetros.

* un diccionario de Python
* una clase de parámetro personalizado
* una clase de datos

Todos estos enfoques funcionan bien y realmente es una cuestión de criterio cuál prefiere. Una desventaja de Python `dict` y una clase personalizada es que ambas son mutables (aunque una clase puede tener propiedades personalizadas donde los usuarios solo pueden acceder a atributos "visibles").  Una clase de datos puede volverse inmutable fácilmente y requiere menos código que una clase personalizada, pero tiene la desventaja de que su sintaxis es un poco menos pitónica. Aquí crearemos una clase de parámetro llamada `Experiment`.  

> ☺️ ¡También usaremos esta reorganización del código para eliminar nuestras variables globales!

## 1. Importaciones

In [ ]:
import numpy as np
import pandas as pd
import simpy
import itertools

## 2. Variables, constantes y valores predeterminados a nivel de cuaderno

Un primer paso útil al configurar un modelo de simulación es definir el caso base o los parámetros tal como están.  Aquí crearemos un conjunto de valores constantes/predeterminados para nuestra clase `Experiment`, pero también podría considerar leerlos desde un archivo.

In [ ]:
# default resources
N_OPERATORS = 13

# default mean inter-arrival time (exp)
MEAN_IAT = 60 / 100

# default service time parameters (triangular)
CALL_LOW = 5.0
CALL_MODE = 7.0
CALL_HIGH = 10.0

# sampling settings
N_STREAMS = 2
DEFAULT_RND_SET = 0

# Boolean switch to display simulation results as the model runs
TRACE = False

# run variables
RESULTS_COLLECTION_PERIOD = 1000

## 3. Clases de distribución

Definiremos dos clases de distribución (`Triangular` y `Exponential`) para encapsular la generación de números aleatorios, los parámetros y las semillas aleatorias utilizadas en el muestreo.  Esto simplifica lo que necesitaremos incluir en la clase `Experiment` y, como veremos más adelante, facilita la variación de distribuciones y parámetros.

In [ ]:
class Triangular:
    """
    Convenience class for the triangular distribution.
    packages up distribution parameters, seed and random generator.
    """

    def __init__(self, low, mode, high, random_seed=None):
        """
        Constructor. Accepts and stores parameters of the triangular dist
        and a random seed.

        Params:
        ------
        low: float
            The smallest values that can be sampled

        mode: float
            The most frequently sample value

        high: float
            The highest value that can be sampled

        random_seed: int | SeedSequence, optional (default=None)
            Used with params to create a series of repeatable samples.
        """
        self.rand = np.random.default_rng(seed=random_seed)
        self.low = low
        self.high = high
        self.mode = mode

    def sample(self, size=None):
        """
        Generate one or more samples from the triangular distribution

        Params:
        --------
        size: int
            the number of samples to return.  If size=None then a single
            sample is returned.

        Returns:
        -------
        float or np.ndarray (if size >=1)
        """
        return self.rand.triangular(self.low, self.mode, self.high, size=size)

In [ ]:
class Exponential:
    """
    Convenience class for the exponential distribution.
    packages up distribution parameters, seed and random generator.
    """

    def __init__(self, mean, random_seed=None):
        """
        Constructor

        Params:
        ------
        mean: float
            The mean of the exponential distribution

        random_seed: int| SeedSequence, optional (default=None)
            A random seed to reproduce samples.  If set to none then a unique
            sample is created.
        """
        self.rand = np.random.default_rng(seed=random_seed)
        self.mean = mean

    def sample(self, size=None):
        """
        Generate a sample from the exponential distribution

        Params:
        -------
        size: int, optional (default=None)
            the number of samples to return.  If size=None then a single
            sample is returned.

        Returns:
        -------
        float or np.ndarray (if size >=1)
        """
        return self.rand.exponential(self.mean, size=size)

## 3. Clase de experimento

Una clase de experimento es útil porque permite configurar y programar fácilmente una gran cantidad de experimentos para que se realicen en un bucle.  Configuramos la clase para que utilice las variables predeterminadas que definimos anteriormente, es decir, por defecto, el modelo refleja el proceso tal como está.  Para ejecutar un nuevo experimento simplemente anulamos los valores predeterminados.

In [ ]:
class Experiment:
    """
    Encapsulates the concept of an experiment 🧪 with the urgent care
    call centre simulation model.

    An Experiment:
    1. Contains a list of parameters that can be left as defaults or varied
    2. Provides a place for the experimentor to record results of a run 
    3. Controls the set & streams of pseudo random numbers used in a run.
    
    """

    def __init__(
        self,
        random_number_set=DEFAULT_RND_SET,
        n_operators=N_OPERATORS,
        mean_iat=MEAN_IAT,
        call_low=CALL_LOW,
        call_mode=CALL_MODE,
        call_high=CALL_HIGH,
        n_streams=N_STREAMS,
    ):
        """
        The init method sets up our defaults.
        """
        # sampling
        self.random_number_set = random_number_set
        self.n_streams = n_streams
        
        # store parameters for the run of the model
        self.n_operators = n_operators
        self.mean_iat = mean_iat
        self.call_low = call_low
        self.call_mode = call_mode
        self.call_high = call_high
        
        # resources: we must init resources after an Environment is created.
        # But we will store a placeholder for transparency
        self.operators = None

        # initialise results to zero
        self.init_results_variables()

        # initialise sampling objects
        self.init_sampling()

    def set_random_no_set(self, random_number_set):
        """
        Controls the random sampling
        Parameters:
        ----------
        random_number_set: int
            Used to control the set of pseudo random numbers used by 
            the distributions in the simulation.
        """
        self.random_number_set = random_number_set
        self.init_sampling()

    def init_sampling(self):
        """
        Create the distributions used by the model and initialise
        the random seeds of each.
        """
        # produce n non-overlapping streams
        seed_sequence = np.random.SeedSequence(self.random_number_set)
        self.seeds = seed_sequence.spawn(self.n_streams)

        # create distributions

        # call inter-arrival times
        self.arrival_dist = Exponential(
            self.mean_iat, random_seed=self.seeds[0]
        )

        # duration of call triage
        self.call_dist = Triangular(
            self.call_low,
            self.call_mode,
            self.call_high,
            random_seed=self.seeds[1],
        )

    def init_results_variables(self):
        """
        Initialise all of the experiment variables used in results
        collection.  This method is called at the start of each run
        of the model
        """
        # variable used to store results of experiment
        self.results = {}
        self.results["waiting_times"] = []

        # total operator usage time for utilisation calculation.
        self.results["total_call_duration"] = 0.0

### 3.1. Crear un experimento predeterminado

Utilizar `Experiment` es muy sencillo.  Por ejemplo, para crear un experimento predeterminado (que utiliza todos los valores de parámetros predeterminados), usaríamos el siguiente código

In [ ]:
env = simpy.Environment()
default_experiment = Experiment()

Debido a la forma en que funciona Python, podemos acceder a todas las variables del experimento desde el objeto `default_scenario`. Por ejemplo, el siguiente código generará un tiempo entre llegadas:

In [ ]:
default_experiment.arrival_dist.sample()

In [ ]:
default_experiment.mean_iat

### 3.2 Creando un experimento con más operadores de llamadas

Para cambiar los parámetros en un experimento solo necesitamos incluir un nuevo valor cuando creamos el `Experiment`.  Por ejemplo si quisiéramos aumentar el número de servidores a 14. Usamos el siguiente código:

In [ ]:
env = simpy.Environment()
extra_server = Experiment(n_operators=14)

In [ ]:
extra_server.n_operators

## 4. Código de modelo modificado

Modificaremos el código del modelo y la lógica que ya hemos desarrollado.  Las funciones de servicio y llegadas ahora aceptarán un argumento `Experiment`.

> Tenga en cuenta que en este punto puede colocar todo el código en un módulo de Python e importar las funciones y clases que necesita en un libro de experimento.

In [ ]:
def trace(msg):
    """
    Turing printing of events on and off.

    Params:
    -------
    msg: str
        string to print to screen.
    """
    if TRACE:
        print(msg)

In [ ]:
def service(identifier, env, args):
    """
    simulates the service process for a call operator

    1. request and wait for a call operator
    2. phone triage (triangular)
    3. exit system

    Params:
    ------

    identifier: int
        A unique identifier for this caller

    env: simpy.Environment
        The current environment the simulation is running in
        We use this to pause and restart the process after a delay.

    args: Experiment
        The settings and input parameters for the current experiment

    """

    # record the time that call entered the queue
    start_wait = env.now

    # MODIFICATION: request an operator - stored in the Experiment
    with args.operators.request() as req:
        yield req

        # record the waiting time for call to be answered
        waiting_time = env.now - start_wait

        # ######################################################################
        # MODIFICATION: store the results for an experiment
        args.results["waiting_times"].append(waiting_time)
        # ######################################################################

        trace(f"operator answered call {identifier} at " + f"{env.now:.3f}")

        # ######################################################################
        # MODIFICATION: the sample distribution is defined by the experiment.
        call_duration = args.call_dist.sample()
        # ######################################################################

        # schedule process to begin again after call_duration
        yield env.timeout(call_duration)

        # update the total call_duration
        args.results["total_call_duration"] += call_duration

        # print out information for patient.
        trace(
            f"call {identifier} ended {env.now:.3f}; "
            + f"waiting time was {waiting_time:.3f}"
        )

In [ ]:
def arrivals_generator(env, args):
    """
    IAT is exponentially distributed

    Parameters:
    ------
    env: simpy.Environment
        The simpy environment for the simulation

    args: Experiment
        The settings and input parameters for the simulation.
    """
    # use itertools as it provides an infinite loop
    # with a counter variable that we can use for unique Ids
    for caller_count in itertools.count(start=1):

        # ######################################################################
        # MODIFICATION:the sample distribution is defined by the experiment.
        inter_arrival_time = args.arrival_dist.sample()
        ########################################################################

        yield env.timeout(inter_arrival_time)

        trace(f"call arrives at: {env.now:.3f}")

        # ######################################################################
        # MODIFICATION: we pass the experiment to the service function
        env.process(service(caller_count, env, args))
        # ######################################################################

## 5. Una función contenedora de ejecución única

In [ ]:
def single_run(experiment, rep=0, rc_period=RESULTS_COLLECTION_PERIOD):
    """
    Perform a single run of the model and return the results

    Parameters:
    -----------

    experiment: Experiment
        The experiment/paramaters to use with model
    """

    # results dictionary.  Each KPI is a new entry.
    run_results = {}

    # reset all result collection variables
    experiment.init_results_variables()

    # set random number set to the replication no.
    # this controls sampling for the run.
    experiment.set_random_no_set(rep)

    # environment is (re)created inside single run
    env = simpy.Environment()

    # we create simpy resource here - this has to be after we
    # create the environment object.
    experiment.operators = simpy.Resource(env, capacity=experiment.n_operators)

    # we pass the experiment to the arrivals generator
    env.process(arrivals_generator(env, experiment))
    env.run(until=rc_period)

    # end of run results: calculate mean waiting time
    run_results["01_mean_waiting_time"] = np.mean(
        experiment.results["waiting_times"]
    )

    # end of run results: calculate mean operator utilisation
    run_results["02_operator_util"] = (
        experiment.results["total_call_duration"]
        / (rc_period * experiment.n_operators)
    ) * 100.0

    # return the results from the run of the model
    return run_results

In [ ]:
TRACE = False
default_scenario = Experiment()
results = single_run(default_scenario)
print(
    f"Mean waiting time: {results['01_mean_waiting_time']:.2f} mins \n"
    + f"Operator Utilisation {results['02_operator_util']:.2f}%"
)

## Múltiples replicaciones

In [ ]:
def multiple_replications(
    experiment, rc_period=RESULTS_COLLECTION_PERIOD, n_reps=5
):
    """
    Perform multiple replications of the model.

    Params:
    ------
    experiment: Experiment
        The experiment/paramaters to use with model

    rc_period: float, optional (default=DEFAULT_RESULTS_COLLECTION_PERIOD)
        results collection period.
        the number of minutes to run the model to collect results

    n_reps: int, optional (default=5)
        Number of independent replications to run.

    Returns:
    --------
    pandas.DataFrame
    """

    # loop over single run to generate results dicts in a python list.
    results = [single_run(experiment, rep, rc_period) for rep in range(n_reps)]

    # format and return results in a dataframe
    df_results = pd.DataFrame(results)
    df_results.index = np.arange(1, len(df_results) + 1)
    df_results.index.name = "rep"
    return df_results

In [ ]:
TRACE = False
default_scenario = Experiment()
results = multiple_replications(default_scenario)
results

## 6. Múltiples experimentos 🧪🧪🧪

La función contenedora `single_run` para el modelo y la clase `Experiment` significan que es muy sencillo ejecutar múltiples experimentos.  Definiremos dos nuevas funciones para ejecutar múltiples experimentos:

* `get_experiments()`: esto devolverá un diccionario de Python que contiene un nombre único para un experimento emparejado con un objeto `Experiment`.
* `run_all_experiments()`: esto recorrerá el diccionario, ejecutará todos los experimentos y devolverá resultados combinados.
* `experiment_summary_frame()`: toma los resultados de cada escenario y formatéalos en una tabla simple.

In [ ]:
def get_experiments():
    """
    Creates a dictionary object containing
    objects of type `Experiment` 🧪 to run.

    Returns:
    --------
    dict
        Contains the experiments for the model
    """
    experiments = {}

    # base (default) case
    experiments["base"] = Experiment()

    # +1 extra capacity
    experiments["operators+1"] = Experiment(
        n_operators=N_OPERATORS + 1,
    )

    return experiments

In [ ]:
def run_all_experiments(experiments, rc_period=RESULTS_COLLECTION_PERIOD):
    """
    Run each of the scenarios for a specified results
    collection period and replications.

    Params:
    ------
    experiments: dict
        dictionary of Experiment objects

    rc_period: float
        model run length

    """
    print("Model experiments:")
    print(f"No. experiments to execute = {len(experiments)}\n")

    experiment_results = {}
    for exp_name, experiment in experiments.items():

        print(f"Running {exp_name}", end=" => ")
        results = multiple_replications(experiment, rc_period)
        print("done.\n")

        # save the results
        experiment_results[exp_name] = results

    print("All experiments are complete.")

    # format the results
    return experiment_results

In [ ]:
# get the experiments
experiments = get_experiments()

# run the scenario analysis
experiment_results = run_all_experiments(experiments)

In [ ]:
experiment_results["operators+1"]

In [ ]:
def experiment_summary_frame(experiment_results):
    """
    Mean results for each performance measure by experiment

    Parameters:
    ----------
    experiment_results: dict
        dictionary of replications.
        Key identifies the performance measure

    Returns:
    -------
    pd.DataFrame
    """
    columns = []
    summary = pd.DataFrame()
    for sc_name, replications in experiment_results.items():
        summary = pd.concat([summary, replications.mean()], axis=1)
        columns.append(sc_name)

    summary.columns = columns
    return summary

In [ ]:
# as well as rounding you may want to rename the cols/rows to
# more readable alternatives.
summary_frame = experiment_summary_frame(experiment_results)
summary_frame.round(2)

## Cargar y múltiples experimentos desde un CSV

La clase `Experiment` proporciona una forma sencilla de ejecutar varios experimentos en un lote. Para hacerlo, podemos crear varias instancias de Experimento, cada una con un conjunto diferente de entradas para el modelo. Luego se ejecutan en un bucle.

### Formatear archivos de experimento

En el formato utilizado aquí, cada fila representa un experimento. La primera columna es un identificador numérico único, la segunda columna es un nombre dado al experimento y las siguientes columnas $n$ representan las variables de entrada opcionales que se pueden pasar a un experimento.

Tenga en cuenta que el método descrito aquí se basa en que los nombres de estas columnas coincidan con los parámetros de entrada de `Experiment`.

   > Pero tenga en cuenta que no es necesario que las columnas estén en el mismo orden que los argumentos del Experimento y no es necesario que sean exhaustivas. Una selección funciona bien.

Por ejemplo, en el call center de atención urgente incluiremos 3 columnas con los nombres:

* n_operadores
* significa_iat

La función `create_example_csv()` crea un archivo que contiene cuatro experimentos que varían estos parámetros.

In [ ]:
def create_example_csv(filename="example_experiments.csv"):
    """
    Create an example CSV file to use in tutorial.
    This creates 4 experiments that varys
    n_operators, and mean_iat.

    Params:
    ------
    filename: str, optional (default='example_experiments.csv')
        The name and path to the CSV file.
    """
    # each column is defined as a seperate list
    names = ["base", "op+1", "high_demand", "combination"]
    operators = [13, 14, 13, 14]
    mean_iat = [0.6, 0.6, 0.55, 0.55]

    # empty dataframe
    df_experiments = pd.DataFrame()

    # create new columns from lists
    df_experiments["experiment"] = names
    df_experiments["n_operators"] = operators
    df_experiments["mean_iat"] = mean_iat

    df_experiments.to_csv(filename, index_label="id")

In [ ]:
create_example_csv()

# load and illustrate results
pd.read_csv("example_experiments.csv", index_col="id")

### Conversión del CSV a instancias de `Experiment`

El código anterior muestra los experimentos al usuario y los almacena como `pd.Dataframe` en la variable `df_experiments`. Convertir las filas en objetos `Experiment` es un proceso de dos pasos.

* Emitimos el `Dataframe` a un diccionario de Python anidado. Cada clave del diccionario es el nombre de un experimento. El valor es otro diccionario donde los pares clave/valor son columnas y sus valores.

* Recorremos las entradas del diccionario y pasamos los parámetros a una nueva instancia de la clase `Experiment`.

La función `create_experiments` implementa ambos pasos. La función devuelve un nuevo diccionario donde los pares clave-valor son la cadena del nombre del experimento y una instancia de `Experiment`.

In [ ]:
def create_experiments(df_experiments):
    """
    Returns dictionary of Experiment objects based on contents of a dataframe

    Params:
    ------
    df_experiments: pandas.DataFrame
        Dataframe of experiments. First two columns are id, name followed by
        variable names.  No fixed width

    Returns:
    --------
    dict
    """
    experiments = {}

    # experiment input parameter dictionary
    exp_dict = df_experiments[df_experiments.columns[1:]].T.to_dict()
    # names of experiments
    exp_names = df_experiments[df_experiments.columns[0]].T.to_list()

    # loop through params and create Experiment objects.
    for name, params in zip(exp_names, exp_dict.values()):
        experiments[name] = Experiment(**params)

    return experiments

In [ ]:
# test of the function

# assume code is run in same directory as example csv file
df_experiment = pd.read_csv("example_experiments.csv", index_col="id")

# convert to dict containing separate Experiment objects
experiments_to_run = create_experiments(df_experiment)

print(type(experiments_to_run))
print(experiments_to_run["op+1"].n_operators)

### Ejecute todos los experimentos y muestre los resultados en una tabla.

Ahora podemos utilizar las funciones `run_all_experiments` y `experiment_summary_frame` para ejecutar y mostrar estos experimentos.

In [ ]:
results = run_all_experiments(experiments_to_run)

# illustrate results dataframe.
results["base"].head(2)

In [ ]:
results["high_demand"].head(2)

In [ ]:
# show results
# further adaptions might include adding units for figures.
experiment_summary_frame(results).round(2)

In [ ]:
experiment_summary_frame(results).round(2).T